# Fine-tune Moonshine-Base-BN: stage 1 (standard Bangladeshi Bangla + phone audio)

**Same pipeline as the .py scripts** in `Fine-tune-moonshine/` (this notebook writes them to disk and runs them),
so the results are identical on Kaggle, Colab, or your friend's GPU PC.

**What stage 1 does**
- trains on **SUBAK.KO** (human-checked standard Bangladeshi Bangla, ~60 h); 60% of the clips are turned into
  **8 kHz phone audio** on the fly, so the model learns the sound of your calls while the text stays correct
- picks the best checkpoint on a SUBAK.KO dev set (half clean, half phone)
- compares the original vs fine-tuned model on: SUBAK.KO test (clean + phone), BanSpeech talk-show/interview/drama
  (phone + clean), BanSpeech **medicine** (phone), news/audiobooks (clean), and **your own labelled calls**

**Which platform?**

| | Kaggle (recommended) | Colab free |
|---|---|---|
| GPU time | ~30 h/week, 12 h per session | a few h/day, varies |
| Browser closed / disconnect | "Save Version → Save & Run All" runs in the background | session dies; re-run all cells, training resumes from Drive |
| Setup | Settings: Accelerator **GPU T4 x2**, Internet **On** (needs a phone-verified account) | Runtime → Change runtime type → **T4 GPU** |

**How to use it:** run the cells from top to bottom. First with `QUICK = True` (~15 min, checks everything works),
then set `QUICK = False` and run everything again for the real training.

## 1. Settings: the only cell you normally change

In [ ]:
QUICK = True                  # True = tiny test run first. Then False = the real run
RUN_NAME = "stage1_subakko_phone"
TRAIN_HOURS = 60              # SUBAK.KO hours (~3 usable h per downloaded file of ~470 MB)
PHONE_SIM_PROB = 0.6          # share of training clips turned into phone audio
NUM_EPOCHS = 3
LEARNING_RATE = 3e-5
BATCH_SIZE, GRAD_ACCUM = 16, 2   # T4: 16 x 2 = 32 clips per update. Out of memory -> 8, 4

# Colab only: keep the prepared data on Google Drive too (~7 GB for 60 h).
# False = rebuilt after every disconnect (~20-40 min); checkpoints are ALWAYS saved to Drive.
DATA_ON_DRIVE = False

## 2. Platform, folders, GPU

- **Colab:** checkpoints, logs and the final model go to Google Drive (`MyDrive/moonshine_ft/runs`), so a
  disconnect loses at most 250 training updates. Downloads stay on the fast local disk.
- **Kaggle:** results go to `/kaggle/working/runs` (kept as the notebook's *Output* when you "Save Version").
  Downloads and prepared data go to `/tmp` (more space, not saved). To **continue** a run in a new session, add
  the previous version's output as an input (*Add Input → Your Work → this notebook*); the cell below copies the
  checkpoints back and training resumes.
- **Your own labelled calls** (the most important test): a folder of short clips, `x.wav` + `x.txt` (correct
  transcript). Colab: upload to `MyDrive/moonshine_ft/own_test_set/`. Kaggle: upload as a **private** dataset
  and add it as input; any input folder named `own_test_set` (or containing `.txt` + audio pairs) is found.
  These are patient calls: keep them private.

In [ ]:
import os, shutil, subprocess, sys, glob
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = "google.colab" in sys.modules or Path("/content").exists() and not ON_KAGGLE
print("platform:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "local")

if ON_KAGGLE:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # use one of the two T4s (the scripts are tested on 1 GPU)
    CODE = Path("/kaggle/working/Fine-tune-moonshine")
    DATA = Path("/tmp/moonshine_ft_data")
    RUNS = Path("/kaggle/working/runs")
    OWN = None
    for d in glob.glob("/kaggle/input/**/own_test_set", recursive=True):
        OWN = Path(d)
    if OWN is None:                                   # any input folder with audio + .txt pairs
        for txt in glob.glob("/kaggle/input/**/*.txt", recursive=True):
            if any(Path(txt).with_suffix(e).exists() for e in (".wav", ".mp3", ".flac", ".m4a", ".ogg")):
                OWN = Path(txt).parent
                break
    # continue a run from a previous version's output
    for old in glob.glob(f"/kaggle/input/**/runs/{RUN_NAME}", recursive=True):
        if not (RUNS / RUN_NAME / "checkpoints").exists() and Path(old, "checkpoints").exists():
            print("copying previous checkpoints from", old)
            shutil.copytree(old, RUNS / RUN_NAME, dirs_exist_ok=True)
elif ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = Path("/content/drive/MyDrive/moonshine_ft")
    CODE = Path("/content/Fine-tune-moonshine")
    DATA = PERSIST / "data" if DATA_ON_DRIVE else Path("/content/ft_data")
    RUNS = PERSIST / "runs"
    OWN = PERSIST / "own_test_set"
else:
    CODE = Path.cwd() / "Fine-tune-moonshine"
    DATA, RUNS, OWN = CODE / "data", CODE / "runs", CODE / "own_test_set"

CODE.mkdir(parents=True, exist_ok=True)
os.chdir(CODE)
print("code:", CODE, "| data:", DATA, "| runs:", RUNS, "| own test set:", OWN)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "NO GPU! Turn the GPU on in the settings.")
print(subprocess.run(["df", "-h", "/tmp" if ON_KAGGLE else "/content" if ON_COLAB else "."],
                     capture_output=True, text=True).stdout)

## 3. Install packages
Colab and Kaggle already have PyTorch with CUDA. We add the exact `transformers` version the scripts were
tested with (Moonshine needs transformers 5.x). Takes ~1-2 min. If pip prints dependency *warnings* about other
preinstalled packages, that is fine.

In [ ]:
!pip -q install "transformers==5.22.2" "accelerate>=1.10" "datasets>=4.0" huggingface_hub jiwer soundfile "librosa>=0.10" tabulate
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__,
      "| cuda", torch.cuda.is_available())

## 4. Write the code files
These cells are copies of the `.py` files (generated by `make_notebook.py`). You do not need to read or change
them here: all settings come from cell 1, which is written to `config_local.py` below. If you want to understand
a step, the docstring at the top of each file explains it (and README.md section 2).

In [ ]:
%%writefile config.py
"""
All settings for the fine-tuning scripts, in one place.

Per-machine changes (paths, batch size for a bigger GPU, ...) go in `config_local.py` next to this file,
so this file stays the same on every machine. Example config_local.py:

    BATCH_SIZE = 32          # 24 GB GPU
    GRAD_ACCUM = 1
    DATA_DIR = Path("/mnt/bigdisk/moonshine_data")

`config_local.py` is executed at the end of this file, so it can override any name below.
The Kaggle / Colab notebooks write their own config_local.py (paths on Google Drive / Kaggle disks).
The `--quick` flag of every script calls apply_quick() (tiny end-to-end test run).
"""

import os
from pathlib import Path

HERE = Path(__file__).resolve().parent
PROJECT_ROOT = HERE.parent

# ---------------------------------------------------------------- run + paths
RUN_NAME = "stage1_subakko_phone"        # change for every new experiment, so results stay separate
DATA_DIR = HERE / "data"                 # processed datasets (processed/), ~7 GB for 60 h
RAW_DIR = None                           # downloaded parquet files. None = DATA_DIR/raw (~11 GB for 60 h)
DELETE_RAW_AFTER_PREP = False            # True = delete downloads after prepare_data (saves disk on Kaggle/Colab)
RUNS_DIR = HERE / "runs"                 # one folder per RUN_NAME: checkpoints, logs, final model, results
MY_AUDIO_DIR = PROJECT_ROOT / "sample-audio"   # your recordings: transcribed before/after (a .txt next to it = scored)
OWN_TEST_DIR = HERE / "own_test_set"     # YOUR labelled call clips: x.wav + x.txt pairs. Scored in baseline + evaluate

BASE_MODEL = "Sanjidh090/moonshine-base-bn"    # or a local folder / a previous final_model (stage 2 starts from stage 1)

# ---------------------------------------------------------------- training data
# A list of sources. Each one: Hugging Face dataset repo, which parquet files, column names, how many hours.
#   phone_sim=True : the clip becomes phone audio with probability PHONE_SIM_PROB (new random version every epoch),
#                    otherwise only the volume changes. Real 8 kHz audio is never phone-simulated again.
#   keep           : optional filter, e.g. {"dialect": ["Sylhet", "Chittagong"]}
# Hours are spread evenly over the files, so more files = more different speakers/books.
SUBAKKO_REPO = "SUST-CSE-Speech/SUBAK.KO"     # Bangladeshi standard Bangla, human-checked, CC-BY-4.0


def subakko_train_files(n):
    """n of the 47 SUBAK.KO train files (~470 MB, ~4.3 h each), spread over the whole set."""
    return [f"data/train-{round(i * 47 / n):05d}-of-00047.parquet" for i in range(n)]


# Stage 1: standard Bangladeshi Bangla. ~30% of the clips are dropped (digits / English letters), so one file
# gives ~3 h: 20 files ~ 60 h (~9.5 GB download). In config_local.py you can change it,
# e.g. TRAIN_SOURCES[0].update(files=subakko_train_files(10), hours=30) for a shorter run.
TRAIN_SOURCES = [
    dict(name="subakko", repo=SUBAKKO_REPO, files=subakko_train_files(20),
         text_col="transcription", hours=60, phone_sim=True),
]

# Stage 2 (later, NOT now): regional dialects. Machine-made labels (Whisper) in dialect spelling, CC-BY-NC.
# Start from the stage 1 model (BASE_MODEL = runs/stage1_subakko_phone/final_model) and keep ~30% SUBAK.KO:
# TRAIN_SOURCES = [
#     dict(name="dialect", repo="niloycste68/Bangali_local_dialect_ASR_HF_Dataset",
#          files=[f"data/train-{i:05d}-of-00027.parquet" for i in range(0, 27, 2)],
#          text_col="sentence", hours=40, phone_sim=True, keep=None, category_col="dialect"),
#     dict(name="subakko", repo=SUBAKKO_REPO,
#          files=[f"data/train-{i:05d}-of-00047.parquet" for i in range(20, 47, 6)],   # other files than stage 1
#          text_col="transcription", hours=17, phone_sim=True),
# ]

# ---------------------------------------------------------------- dev + test data (never trained on)
# dev  = SUBAK.KO validation clips, half clean + half phone-simulated. Picks the best checkpoint.
# tests: SUBAK.KO test (clean + the same clips phone-simulated) and BanSpeech domains (Bangladeshi broadcast)
SUBAKKO_VALID_FILES = ["data/validation-00000-of-00005.parquet"]
SUBAKKO_TEST_FILES = ["data/test-00000-of-00005.parquet", "data/test-00003-of-00005.parquet"]
BANSPEECH_REPO = "SUST-CSE-Speech/banspeech"
BANSPEECH_SPONT = ["political_talkshow", "celebrity_interview", "drama_series"]   # free conversation
BANSPEECH_MEDICINE = ["medicine"]                                                 # medical words
BANSPEECH_CLEAN = ["television_news", "audio_books"]
TEST_CLIPS_PER_SPLIT = 300
DEV_CLIPS = 300

# ---------------------------------------------------------------- filters for training clips
MIN_SEC, MAX_SEC = 1.0, 20.0
MAX_LABEL_TOKENS = 180        # the decoder has 194 positions
DROP_LATIN = True             # our rule: English words are written in Bangla script
DROP_DIGITS = True            # our rule: numbers are written as words

# ---------------------------------------------------------------- augmentation
PHONE_SIM_PROB = 0.6          # 60% phone audio (the target), 40% clean (so clean audio is not forgotten)

# ---------------------------------------------------------------- training
LEARNING_RATE = 3e-5
NUM_EPOCHS = 3
MAX_STEPS = -1                # -1 = use NUM_EPOCHS; > 0 overrides
BATCH_SIZE = 16               # T4 16 GB: 16. 24 GB GPU: 32 (then GRAD_ACCUM = 1)
GRAD_ACCUM = 2                # effective batch = BATCH_SIZE * GRAD_ACCUM = 32
EVAL_BATCH_SIZE = 32
WARMUP = 0.10
WEIGHT_DECAY = 0.01
EVAL_STEPS = 250              # dev evaluation + checkpoint every N updates
SAVE_TOTAL_LIMIT = 2
PRECISION = "auto"            # auto (bf16 if the GPU supports it, else fp16) | fp16 | bf16 | fp32
NUM_WORKERS = 2               # dataloader worker processes (augmentation runs there)
SEED = 42

QUICK = False


def apply_quick():
    """Tiny end-to-end run (~10-15 min on a GPU) to check that everything works."""
    global QUICK, RUN_NAME, TRAIN_SOURCES, SUBAKKO_TEST_FILES, TEST_CLIPS_PER_SPLIT, DEV_CLIPS, MAX_STEPS, EVAL_STEPS
    QUICK = True
    RUN_NAME = "quick_test"
    TRAIN_SOURCES = [dict(s, files=s["files"][:1], hours=min(s["hours"], 0.5)) for s in TRAIN_SOURCES]
    SUBAKKO_TEST_FILES = SUBAKKO_TEST_FILES[:1]
    TEST_CLIPS_PER_SPLIT = min(TEST_CLIPS_PER_SPLIT, 30)
    DEV_CLIPS = min(DEV_CLIPS, 40)
    MAX_STEPS = 60
    EVAL_STEPS = 30


def processed_dir() -> Path:
    """Prepared datasets. Shared by all runs that use the same data settings."""
    return Path(DATA_DIR) / ("processed_quick" if QUICK else "processed")


def raw_dir() -> Path:
    return Path(RAW_DIR) if RAW_DIR else Path(DATA_DIR) / "raw"


def run_dir() -> Path:
    return Path(RUNS_DIR) / RUN_NAME


def data_settings() -> dict:
    """Everything that changes the prepared data. If it changes, prepare_data.py rebuilds."""
    return {k: (str(v) if isinstance(v, Path) else v) for k, v in dict(
        TRAIN_SOURCES=TRAIN_SOURCES, SUBAKKO_REPO=SUBAKKO_REPO, SUBAKKO_VALID_FILES=SUBAKKO_VALID_FILES,
        SUBAKKO_TEST_FILES=SUBAKKO_TEST_FILES, BANSPEECH_REPO=BANSPEECH_REPO, BANSPEECH_SPONT=BANSPEECH_SPONT,
        BANSPEECH_MEDICINE=BANSPEECH_MEDICINE, BANSPEECH_CLEAN=BANSPEECH_CLEAN,
        TEST_CLIPS_PER_SPLIT=TEST_CLIPS_PER_SPLIT, DEV_CLIPS=DEV_CLIPS, MIN_SEC=MIN_SEC, MAX_SEC=MAX_SEC,
        MAX_LABEL_TOKENS=MAX_LABEL_TOKENS, DROP_LATIN=DROP_LATIN, DROP_DIGITS=DROP_DIGITS,
        BASE_MODEL=str(BASE_MODEL), SEED=SEED,
    ).items()}


# ---------------------------------------------------------------- per-machine overrides
_local = HERE / "config_local.py"
if _local.exists():
    exec(compile(_local.read_text(encoding="utf-8"), str(_local), "exec"))

In [ ]:
%%writefile common.py
"""
Shared code for the fine-tuning scripts: setup/logging, text cleaning, audio decoding, phone simulation,
model loading, the training-batch collator, and evaluation (transcribe + WER/CER).
"""

import argparse
import io
import os
import random
import re
import sys
import tempfile
import time
import unicodedata
from datetime import datetime
from pathlib import Path

import numpy as np

import config as C

SR = 16000
START_ID, EOS_ID, PAD_ID = 2, 3, 0      # [CLS], [SEP], [PAD]. Checked against the model; see README


# =============================================================== setup / logging
class _Tee:
    """Write everything printed to the console into a log file too."""

    def __init__(self, stream, fh):
        self.stream, self.fh = stream, fh

    def write(self, s):
        self.stream.write(s)
        self.fh.write(s)
        self.fh.flush()

    def flush(self):
        self.stream.flush()
        self.fh.flush()

    def isatty(self):
        return False

    def __getattr__(self, name):
        return getattr(self.stream, name)


def load_dotenv():
    """Read HF_TOKEN etc. from the project's .env file (if the variable is not already set)."""
    for env_file in (C.HERE / ".env", C.PROJECT_ROOT / ".env"):
        if env_file.exists():
            for line in env_file.read_text(encoding="utf-8").splitlines():
                if "=" in line and not line.lstrip().startswith("#"):
                    k, v = line.split("=", 1)
                    os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))


def base_parser(description):
    p = argparse.ArgumentParser(description=description)
    p.add_argument("--quick", action="store_true", help="tiny end-to-end test run (do this first)")
    p.add_argument("--run-name", help="override RUN_NAME from config.py")
    return p


def setup(args, step):
    """Apply --quick / --run-name, create folders, log to runs/<run>/logs/<step>_<time>.log, seed."""
    if getattr(args, "quick", False) and not C.QUICK:
        C.apply_quick()
    if getattr(args, "run_name", None):
        C.RUN_NAME = args.run_name
    for d in (C.raw_dir(), C.processed_dir(), C.run_dir() / "logs"):
        d.mkdir(parents=True, exist_ok=True)
    if not isinstance(sys.stdout, _Tee):
        log = C.run_dir() / "logs" / f"{step}_{datetime.now():%Y%m%d_%H%M%S}.log"
        fh = open(log, "a", encoding="utf-8")
        sys.stdout, sys.stderr = _Tee(sys.stdout, fh), _Tee(sys.stderr, fh)
        print(f"log file: {log}")
    for s in (sys.stdout, sys.stderr):
        try:
            s.stream.reconfigure(encoding="utf-8")   # Bangla text on a Windows console
        except Exception:
            pass
    load_dotenv()
    random.seed(C.SEED)
    np.random.seed(C.SEED)
    import torch
    torch.manual_seed(C.SEED)
    print(f"=== {step} | run={C.RUN_NAME} | quick={C.QUICK} | {datetime.now():%Y-%m-%d %H:%M:%S}")


def device():
    import torch
    return "cuda" if torch.cuda.is_available() else "cpu"


def precision():
    """'bf16' | 'fp16' | 'fp32'. bf16 (Ampere+ GPUs: RTX 30xx/40xx, A100, ...) is more stable than fp16."""
    import torch
    if device() != "cuda":
        return "fp32"
    if C.PRECISION != "auto":
        return C.PRECISION
    return "bf16" if torch.cuda.is_bf16_supported() else "fp16"


# =============================================================== text
_TAGS = re.compile(r"\[[^\]]*\]|<[^>]*>|\{[^}]*\}")
_PUNCT = re.compile(r"[।॥,\.\?!;:\"'‘’“”\-–—\(\)/\\…*_~`|=+#@&%^]")
_ZW = re.compile(r"[​‌‍﻿]")
_LATIN = re.compile(r"[A-Za-z]")
_DIGIT = re.compile(r"[0-9০-৯]")


def normalize_text(s):
    """One cleaning rule for training labels AND scoring: NFC, no tags, no punctuation, single spaces."""
    s = unicodedata.normalize("NFC", s or "")
    s = _TAGS.sub(" ", s)
    s = _ZW.sub("", s)
    s = _PUNCT.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()


def check_clip(tokenizer, text, dur):
    """Returns (clean text, label ids) or (None, reason why the clip is dropped)."""
    t = normalize_text(text)
    if not t:
        return None, "empty text"
    if dur < C.MIN_SEC:
        return None, "too short"
    if dur > C.MAX_SEC:
        return None, "too long"
    if C.DROP_LATIN and _LATIN.search(t):
        return None, "latin letters"
    if C.DROP_DIGITS and _DIGIT.search(t):
        return None, "digits"
    wps = len(t.split()) / dur
    if dur > 3 and (wps > 6 or wps < 0.3):
        return None, "words/sec implausible"
    ids = tokenizer(t, add_special_tokens=False)["input_ids"] + [EOS_ID]   # labels: tokens + [SEP], no [CLS]
    if len(ids) > C.MAX_LABEL_TOKENS:
        return None, "too many tokens"
    if tokenizer.unk_token_id in ids:
        return None, "unknown token"
    return t, ids


# =============================================================== audio
def decode_audio(b):
    """Audio file bytes -> (16 kHz mono float32, native sample rate)."""
    import librosa
    import soundfile as sf
    try:
        y, sr = sf.read(io.BytesIO(b), dtype="float32", always_2d=True)
        y = y.mean(axis=1)
    except Exception:
        with tempfile.NamedTemporaryFile(suffix=".bin", delete=False) as f:
            f.write(b)
            tmp = f.name
        try:
            y, sr = librosa.load(tmp, sr=None, mono=True)
        finally:
            os.remove(tmp)
    if sr != SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=SR)
    return y.astype(np.float32), int(sr)


def to_int16(y):
    return np.clip(y * 32767, -32768, 32767).astype(np.int16)


_SOS_PHONE = None


def phone_sim(y, rng):
    """16 kHz clip -> phone-line version (returned at 16 kHz, like our real calls after upsampling)."""
    global _SOS_PHONE
    from scipy.signal import butter, resample_poly, sosfilt
    if _SOS_PHONE is None:
        _SOS_PHONE = butter(4, [300, 3400], btype="bandpass", fs=8000, output="sos")
    y8 = resample_poly(y, 1, 2)                                   # 16 -> 8 kHz: nothing above 4 kHz survives
    y8 = sosfilt(_SOS_PHONE, y8)                                  # phone band 300-3400 Hz
    y8 = y8 / (np.abs(y8).max() + 1e-9) * rng.uniform(0.3, 0.9)   # random level
    mu = 255.0                                                    # G.711 mu-law codec (8 bit)
    c = np.sign(y8) * np.log1p(mu * np.abs(y8)) / np.log1p(mu)
    c = np.round((c + 1) / 2 * mu) / mu * 2 - 1
    y8 = np.sign(c) * ((1 + mu) ** np.abs(c) - 1) / mu
    snr_db = rng.uniform(15, 35)                                  # line / room noise
    p = np.mean(y8 ** 2) + 1e-12
    y8 = y8 + rng.normal(0, np.sqrt(p / 10 ** (snr_db / 10)), len(y8))
    return np.clip(resample_poly(y8, 2, 1), -1, 1).astype(np.float32)


# =============================================================== downloads / parquet
def download(repo, files):
    """Download dataset files to data/raw/<repo> (skips files that are already there)."""
    from huggingface_hub import hf_hub_download
    out = []
    for f in files:
        try:
            p = hf_hub_download(repo_id=repo, filename=f, repo_type="dataset",
                                local_dir=str(C.raw_dir() / repo.replace("/", "__")))
        except Exception as e:
            msg = str(e)
            if "gated" in msg.lower() or "401" in msg or "403" in msg:
                raise SystemExit(
                    f"\nNo access to {repo}.\n  1. Open https://huggingface.co/datasets/{repo} and accept the conditions"
                    f"\n  2. Put HF_TOKEN=<your read token> in the project .env (or `huggingface-cli login`)\n") from e
            raise
        out.append(p)
        print(f"  ok {f} ({os.path.getsize(p) / 1e6:.0f} MB)")
    return out


def find_audio_col(schema):
    import pyarrow as pa
    for f in schema:
        if pa.types.is_struct(f.type) and "bytes" in [c.name for c in f.type]:
            return f.name
    raise ValueError(f"no audio column (struct with 'bytes') in {schema.names}")


def string_cols(schema):
    import pyarrow as pa
    return [f.name for f in schema if pa.types.is_string(f.type) or pa.types.is_large_string(f.type)]


def iter_clips(paths, audio_col, text_col, keep=None, category_col=None):
    """Yields (16 kHz float audio, native sr, raw text, category) from parquet files, one row group at a time."""
    import pyarrow.parquet as pq
    cols = list(dict.fromkeys([audio_col, text_col] + ([category_col] if category_col else []) + list(keep or {})))
    for p in paths:
        pf = pq.ParquetFile(p)
        for g in range(pf.num_row_groups):
            for r in pf.read_row_group(g, columns=cols).to_pylist():
                if keep and any(r.get(k) not in v for k, v in keep.items()):
                    continue
                b = (r[audio_col] or {}).get("bytes")
                if not b:
                    continue
                y, sr = decode_audio(b)
                yield y, sr, r[text_col], (str(r.get(category_col)) if category_col else "")


# =============================================================== model
def load_tokenizer():
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(str(C.BASE_MODEL))
    assert tok.convert_ids_to_tokens([PAD_ID, START_ID, EOS_ID]) == ["[PAD]", "[CLS]", "[SEP]"]
    return tok


def fixed_generation_config():
    """The model's saved generation_config.json has wrong ids (1/2/2). These are the real ones."""
    from transformers import GenerationConfig
    return GenerationConfig(decoder_start_token_id=START_ID, bos_token_id=START_ID,
                            eos_token_id=EOS_ID, pad_token_id=PAD_ID)


def load_model(path=None):
    import torch
    from transformers import AutoModelForSpeechSeq2Seq
    m = AutoModelForSpeechSeq2Seq.from_pretrained(str(path or C.BASE_MODEL), dtype=torch.float32)
    m.generation_config = fixed_generation_config()
    return m.to(device())


# =============================================================== datasets on disk
def features():
    from datasets import Features, Value
    try:
        from datasets import List as _List
    except ImportError:
        from datasets import Sequence as _List
    return Features({
        "audio": _List(Value("int16")), "text": Value("string"), "labels": _List(Value("int32")),
        "length": Value("int32"), "native_sr": Value("int32"), "source": Value("string"),
        "category": Value("string"), "augment": Value("int8"),
    })


def load_eval_sets():
    """{name: [(16 kHz float audio, text), ...]} for every saved test set."""
    from datasets import load_from_disk
    root = C.processed_dir() / "tests"
    if not root.exists():
        raise SystemExit(f"No test sets in {root}. Run prepare_data.py first.")
    sets = {}
    for d in sorted(root.iterdir()):
        ds = load_from_disk(str(d)).with_format("numpy", columns=["audio", "text"])
        sets[d.name] = [(np.asarray(r["audio"], np.float32) / 32768.0, str(r["text"])) for r in ds]
    return sets


# =============================================================== training batches
class SpeechCollator:
    """
    Builds one training batch:
      1. 16-bit audio -> float
      2. augment: training clips with augment=1 become phone audio with probability phone_sim_prob (a new
         random version every epoch), otherwise only their volume changes; real 8 kHz recordings are never
         phone-simulated again; augment=2 clips only get a volume change; dev/test clips (augment=0) stay as they are
      3. pad audio to the longest clip (multiple of 320 samples) + attention mask (1 = audio, 0 = padding)
      4. pad labels with -100 (ignored by the loss)
    """

    def __init__(self, phone_sim_prob, seed=0):
        self.phone_sim_prob, self.seed, self.pid, self.rng = phone_sim_prob, seed, None, None

    def __call__(self, feats):
        import torch
        if self.pid != os.getpid():                   # each dataloader worker gets its own random stream
            self.pid = os.getpid()
            self.rng = np.random.default_rng([self.seed, self.pid, time.time_ns() % (2 ** 32)])
        waves = []
        for f in feats:
            y = np.asarray(f["audio"], dtype=np.float32) / 32768.0
            aug = int(f["augment"])
            if aug == 1 and int(f["native_sr"]) > 8000 and self.rng.random() < self.phone_sim_prob:
                y = phone_sim(y, self.rng)
            elif aug in (1, 2):
                y = np.clip(y * self.rng.uniform(0.6, 1.4), -1, 1)
            waves.append(y)
        n = max(len(w) for w in waves)
        n += (-n) % 320
        x = np.zeros((len(waves), n), np.float32)
        am = np.zeros((len(waves), n), np.int64)
        for i, w in enumerate(waves):
            x[i, :len(w)] = w
            am[i, :len(w)] = 1
        L = max(len(f["labels"]) for f in feats)
        lab = np.full((len(feats), L), -100, np.int64)
        for i, f in enumerate(feats):
            lab[i, :len(f["labels"])] = f["labels"]
        return {"input_values": torch.from_numpy(x), "attention_mask": torch.from_numpy(am),
                "labels": torch.from_numpy(lab)}


# =============================================================== evaluation
def transcribe(model, tokenizer, arrays, beams=5, batch_size=16):
    """Production decoding, same as moonshine_base_bn/moonshine_bn.py: beam 5, no_repeat_ngram 3, rep. penalty 1.2."""
    import torch
    dev, prec = device(), precision()
    dtype = {"bf16": torch.bfloat16, "fp16": torch.float16}.get(prec)
    model.eval()
    order = np.argsort([-len(a) for a in arrays])
    out = [None] * len(arrays)
    with torch.no_grad():
        for s in range(0, len(order), batch_size):
            idx = order[s:s + batch_size]
            batch = [arrays[i] for i in idx]
            n = max(len(a) for a in batch)
            n += (-n) % 320
            x = np.zeros((len(batch), n), np.float32)
            am = np.zeros((len(batch), n), np.int64)
            for j, a in enumerate(batch):
                x[j, :len(a)] = a
                am[j, :len(a)] = 1
            max_new = min(int(n / SR * 8) + 16, 190)
            with torch.autocast("cuda", dtype=dtype or torch.float16, enabled=dtype is not None):
                ids = model.generate(torch.from_numpy(x).to(dev), attention_mask=torch.from_numpy(am).to(dev),
                                     max_new_tokens=max_new, num_beams=beams, no_repeat_ngram_size=3,
                                     repetition_penalty=1.2, decoder_start_token_id=START_ID,
                                     eos_token_id=EOS_ID, pad_token_id=PAD_ID)
            for j, i in enumerate(idx):
                out[i] = normalize_text(tokenizer.decode(ids[j], skip_special_tokens=True))
    return out


def score(refs, hyps):
    import jiwer
    pairs = [(r, h if h else "<empty>") for r, h in zip(refs, hyps) if r]
    r, h = zip(*pairs)
    return jiwer.wer(list(r), list(h)), jiwer.cer(list(r), list(h))


def evaluate_sets(model, tokenizer, sets, label, beams=5):
    """Returns (DataFrame with WER/CER per test set, {set name: hypotheses})."""
    import pandas as pd
    rows, hyps_all = [], {}
    for name, data in sets.items():
        t0 = time.time()
        hyps = transcribe(model, tokenizer, [y for y, _ in data], beams=beams)
        w, c = score([t for _, t in data], hyps)
        hyps_all[name] = hyps
        rows.append({"model": label, "test_set": name, "clips": len(data), "WER": round(w, 4),
                     "CER": round(c, 4), "sec": round(time.time() - t0, 1)})
        print(f"  {label:10s} {name:22s} WER {w:.3f}  CER {c:.3f}  ({time.time() - t0:.0f}s)")
    return pd.DataFrame(rows), hyps_all


def transcribe_file(model, tokenizer, path, chunk_sec=20, beams=5):
    """Whole audio file in 20 s chunks, like moonshine_base_bn/test_model.py."""
    import librosa
    y, _ = librosa.load(str(path), sr=SR, mono=True)
    size = int(chunk_sec * SR)
    chunks = [c for c in (y[i:i + size] for i in range(0, len(y), size)) if len(c) >= 1600] or [y]
    return " ".join(h for h in transcribe(model, tokenizer, chunks, beams=beams, batch_size=4) if h)


# =============================================================== your own labelled calls
AUDIO_EXTS = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".opus", ".webm"}


def own_test_files():
    """[(audio path, reference text)] from OWN_TEST_DIR: every audio file that has a .txt with the same name."""
    d = Path(C.OWN_TEST_DIR)
    if not d.is_dir():
        return []
    pairs = []
    for p in sorted(d.rglob("*")):
        if p.suffix.lower() in AUDIO_EXTS and p.with_suffix(".txt").exists():
            ref = normalize_text(p.with_suffix(".txt").read_text(encoding="utf-8"))
            if ref:
                pairs.append((p, ref))
    return pairs


def evaluate_own(model, tokenizer, label, pairs):
    """Score a model on your own call clips. Long files are transcribed in 20 s chunks."""
    t0 = time.time()
    hyps = [transcribe_file(model, tokenizer, p) for p, _ in pairs]
    w, c = score([r for _, r in pairs], hyps)
    print(f"  {label:10s} {'own_test_set':22s} WER {w:.3f}  CER {c:.3f}  ({len(pairs)} files, {time.time() - t0:.0f}s)")
    return {"model": label, "test_set": "own_test_set", "clips": len(pairs), "WER": round(w, 4), "CER": round(c, 4),
            "sec": round(time.time() - t0, 1)}, hyps

In [ ]:
%%writefile prepare_data.py
"""
STEP 1: download, inspect and prepare all data.

  python prepare_data.py --inspect     # download one training file, print columns/examples, save example wavs, stop
  python prepare_data.py               # build everything (skips if already built with the same settings)
  python prepare_data.py --force       # rebuild even if it exists

Output (data/processed/ or data/processed_quick/):
  train/             training clips from TRAIN_SOURCES (config.py), shuffled
  dev/               SUBAK.KO validation clips, half clean + half phone-simulated (picks the best checkpoint)
  tests/<name>/      test sets (never trained on), phone simulation already applied where the name says so
  examples/          wav files + spectrogram so you can LISTEN to the data and the phone simulation
  data_report.json   hours per source, dropped clips and why
  prep_settings.json the settings used; if config.py changes, the next run rebuilds automatically

What happens to each clip: decode -> mono -> 16 kHz -> clean the text -> filters -> tokenize -> stored as
16-bit audio on disk (Arrow), so the training set does not have to fit in RAM.
"""

import io
import json
import shutil
import time
from collections import Counter
from pathlib import Path

import numpy as np

import common as K
import config as C

DROPS = Counter()


def inspect_source(src, paths):
    """Print what the first training file contains and write example wavs (original + phone-simulated)."""
    import pyarrow.parquet as pq
    import soundfile as sf

    pf = pq.ParquetFile(paths[0])
    schema = pf.schema_arrow
    audio_col = K.find_audio_col(schema)
    str_cols = K.string_cols(schema)
    print(f"\n================ {src['name']}: what is in the data ================")
    print("all columns  :", schema.names)
    print("audio column :", audio_col, "| text column:", src["text_col"])
    if src["text_col"] not in schema.names:
        raise SystemExit(f"text_col '{src['text_col']}' is not a column of {paths[0]}")
    peek = pf.read_row_group(0, columns=str_cols).to_pandas()
    print("\n--- 5 example texts (raw -> cleaned training label) ---")
    for t in peek[src["text_col"]].head(5):
        print("  raw  :", str(t)[:160])
        print("  label:", K.normalize_text(t)[:160])
    for c in str_cols:
        vc = peek[c].value_counts()
        if 1 < len(vc) <= 30:
            print(f"\n[{c}]\n{vc.to_string()}")

    rows = pf.read_row_group(0, columns=[audio_col, src["text_col"]]).to_pylist()[:200]
    srs, durs = Counter(), []
    for r in rows:
        b = (r[audio_col] or {}).get("bytes")
        if b:
            info = sf.info(io.BytesIO(b))
            srs[info.samplerate] += 1
            durs.append(info.duration)
    print(f"\nnative sample rates (first {len(rows)} clips): {dict(srs)}   (8000 = telephone recordings)")
    print(f"duration: mean {np.mean(durs):.1f} s, max {np.max(durs):.1f} s")

    ex = C.processed_dir() / "examples"
    ex.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(0)
    for i, r in enumerate(rows[:3]):
        y, sr = K.decode_audio(r[audio_col]["bytes"])
        sf.write(ex / f"{src['name']}_{i}_{sr}Hz.wav", y, K.SR)
        sf.write(ex / f"{src['name']}_{i}_phone_sim.wav", K.phone_sim(y, rng), K.SR)
        (ex / f"{src['name']}_{i}.txt").write_text(str(r[src["text_col"]]), encoding="utf-8")
    print(f"\nexample audio written to {ex} (original + phone-simulated): listen to them!")
    print("=" * 68 + "\n")


def gen_rows(paths, audio_col, text_col, source, max_hours, keep=None, category_col=None, augment=1, cfg=""):
    """Yields training rows. The hours are spread evenly over the files (more speakers, more books)."""
    tokenizer = K.load_tokenizer()
    per_file = max_hours * 3600 / len(paths)
    for p in paths:
        total = 0.0
        for y, sr, text, cat in K.iter_clips([p], audio_col, text_col, keep, category_col):
            t, ids = K.check_clip(tokenizer, text, len(y) / K.SR)
            if t is None:
                DROPS[f"{source}: {ids}"] += 1
                continue
            yield {"audio": K.to_int16(y), "text": t, "labels": ids, "length": len(y), "native_sr": sr,
                   "source": source, "category": cat, "augment": augment}
            total += len(y) / K.SR
            if total >= per_file:
                break


def build(name, **kw):
    from datasets import Dataset
    kw["cfg"] = json.dumps(C.data_settings(), sort_keys=True)     # part of the cache key
    return Dataset.from_generator(gen_rows, gen_kwargs=kw, features=K.features(),
                                  cache_dir=str(C.processed_dir() / "_cache" / name))


def hours(ds):
    return float(np.sum(ds["length"])) / K.SR / 3600


def summary(ds, by):
    import pandas as pd
    df = pd.DataFrame({by: ds[by], "sec": np.asarray(ds["length"]) / K.SR})
    return df.groupby(by)["sec"].agg(clips="count", hours=lambda s: s.sum() / 3600, mean_sec="mean").round(2)


def load_rows(tokenizer, paths, audio_col, text_col):
    """All usable clips of (small) files: [(audio, text, native sr)]."""
    rows = []
    for y, sr, text, _ in K.iter_clips(paths, audio_col, text_col):
        t, _ = K.check_clip(tokenizer, text, len(y) / K.SR)
        if t is not None:
            rows.append((y, t, sr))
    return rows


def load_random_rows(tokenizer, paths, text_col, n, seed):
    """n random usable clips from (big) files, decoding only the clips that are picked."""
    import pyarrow as pa
    import pyarrow.parquet as pq
    table = pa.concat_tables([pq.read_table(p, columns=["audio", text_col]) for p in paths])
    texts = table.column(text_col).to_pylist()
    audio = table.column("audio")
    rows = []
    for i in np.random.default_rng(seed).permutation(len(texts)):
        b = (audio[int(i)].as_py() or {}).get("bytes")
        if not b:
            continue
        y, sr = K.decode_audio(b)
        t, _ = K.check_clip(tokenizer, texts[int(i)], len(y) / K.SR)
        if t is not None:
            rows.append((y, t, sr))
            if len(rows) >= n:
                break
    return rows


def simulate(row, seed):
    """(audio, text, sr) -> the same clip through a simulated phone line (fixed seed = identical every run)."""
    y, t, sr = row
    return (K.phone_sim(y, np.random.default_rng(seed)) if sr > 8000 else y), t, sr


def sample_rows(rows, n, seed, phone):
    idx = [int(i) for i in np.random.default_rng(seed).permutation(len(rows))][:n]
    return [simulate(rows[i], seed * 100003 + i) if phone else rows[i] for i in idx]


def save_rows(tokenizer, rows, path, source):
    from datasets import Dataset
    Dataset.from_dict({
        "audio": [K.to_int16(y) for y, _, _ in rows], "text": [t for _, t, _ in rows],
        "labels": [tokenizer(t, add_special_tokens=False)["input_ids"] + [K.EOS_ID] for _, t, _ in rows],
        "length": [len(y) for y, _, _ in rows], "native_sr": [sr for _, _, sr in rows],
        "source": [source] * len(rows), "category": [""] * len(rows), "augment": [0] * len(rows),
    }, features=K.features()).save_to_disk(str(path))
    print(f"  {path.name:24s} {len(rows):5d} clips  {sum(len(r[0]) for r in rows) / K.SR / 3600:.2f} h")


def phone_sim_picture(rows, ex_dir):
    """Spectrogram: clean vs phone-simulated vs a real call (if one is in MY_AUDIO_DIR)."""
    import librosa
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import soundfile as sf

    y_clean = rows[0][0]
    y_phone = K.phone_sim(y_clean, np.random.default_rng(0))
    sf.write(ex_dir / "banspeech_clean.wav", y_clean, K.SR)
    sf.write(ex_dir / "banspeech_phone_sim.wav", y_phone, K.SR)
    panels = [("clean", y_clean), ("phone-simulated", y_phone)]
    my = Path(C.MY_AUDIO_DIR)
    real = sorted(my.glob("*call*")) if my.is_dir() else []
    if real:
        y_real, _ = librosa.load(str(real[0]), sr=K.SR, offset=10, duration=6)
        panels.append(("real call: " + real[0].name, y_real))
    fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 3.2))
    for ax, (title, y) in zip(np.atleast_1d(axes), panels):
        S = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=512)), ref=np.max)
        ax.imshow(S, origin="lower", aspect="auto", cmap="magma", extent=[0, len(y) / K.SR, 0, K.SR / 2])
        ax.axhline(3400, color="cyan", lw=0.8, ls="--")
        ax.set_title(title, fontsize=9)
        ax.set_xlabel("seconds")
        ax.set_ylabel("Hz")
    plt.tight_layout()
    plt.savefig(ex_dir / "phone_sim_spectrogram.png", dpi=110)
    plt.close(fig)
    print(f"  spectrogram + wavs: {ex_dir}")


def main(args):
    import pyarrow.parquet as pq
    from datasets import concatenate_datasets
    from huggingface_hub import list_repo_files

    out = C.processed_dir()
    settings_file = out / "prep_settings.json"
    settings = C.data_settings()
    if not args.force and not args.inspect and settings_file.exists():
        if json.loads(settings_file.read_text(encoding="utf-8")) == settings:
            print(f"Data already prepared with these settings in {out}. Use --force to rebuild.")
            return
        print("Data settings changed since the last build -> rebuilding.")
    t_start = time.time()

    if args.inspect:
        src = C.TRAIN_SOURCES[0]
        inspect_source(src, K.download(src["repo"], src["files"][:1]))
        return

    for d in ("train", "dev", "tests", "_cache"):
        shutil.rmtree(out / d, ignore_errors=True)
    settings_file.unlink(missing_ok=True)
    tokenizer = K.load_tokenizer()
    report = {"dropped": {}}

    print("\n[1/4] Training data")
    parts = []
    for n, src in enumerate(C.TRAIN_SOURCES):
        print(f"\n--- {src['name']}: {len(src['files'])} files from {src['repo']}, up to {src['hours']} h")
        paths = K.download(src["repo"], src["files"])
        if n == 0:
            inspect_source(src, paths)
        DROPS.clear()
        t0 = time.time()
        ds = build(src["name"], paths=paths, audio_col=K.find_audio_col(pq.ParquetFile(paths[0]).schema_arrow),
                   text_col=src["text_col"], source=src["name"], max_hours=src["hours"], keep=src.get("keep"),
                   category_col=src.get("category_col"), augment=1 if src.get("phone_sim", True) else 2)
        print(f"  built in {(time.time() - t0) / 60:.1f} min: {len(ds)} clips, {hours(ds):.1f} h")
        if src.get("category_col"):
            print(summary(ds, "category"))
        print(summary(ds, "native_sr"))
        print("  dropped:", dict(DROPS))
        report["dropped"].update(DROPS)
        if hours(ds) < 0.9 * src["hours"]:
            print(f"  NOTE: only {hours(ds):.1f} of {src['hours']} h. Add more files to this source for the full amount.")
        parts.append(ds)
    train = concatenate_datasets(parts).shuffle(seed=C.SEED)
    train.save_to_disk(str(out / "train"))
    report["train"] = summary(train, "source").reset_index().to_dict("records")
    report["train_by_native_sr"] = summary(train, "native_sr").reset_index().to_dict("records")
    report["train_hours"] = round(hours(train), 2)
    report["train_clips"] = len(train)
    print(f"\n  TRAIN: {len(train)} clips, {hours(train):.1f} h -> {out / 'train'}")
    del train, parts
    shutil.rmtree(out / "_cache", ignore_errors=True)

    print("\n[2/4] Dev set (SUBAK.KO validation: half clean, half phone)")
    dev = load_random_rows(tokenizer, K.download(C.SUBAKKO_REPO, C.SUBAKKO_VALID_FILES), "transcription",
                           C.DEV_CLIPS, seed=1)
    half = len(dev) // 2
    dev = dev[:half] + [simulate(r, 1000 + i) for i, r in enumerate(dev[half:])]
    save_rows(tokenizer, dev, out / "dev", "dev_subakko_mixed")
    del dev

    print("\n[3/4] Test sets")
    sk_test = load_random_rows(tokenizer, K.download(C.SUBAKKO_REPO, C.SUBAKKO_TEST_FILES), "transcription",
                               C.TEST_CLIPS_PER_SPLIT, seed=2)
    bs_files = {f.split("/")[-1].split("-")[0]: f for f in list_repo_files(C.BANSPEECH_REPO, repo_type="dataset")
                if f.startswith("data/") and f.endswith(".parquet")}
    cache = {}

    def banspeech(splits, phone, seed=7):
        rows = []
        for k, s in enumerate(splits):
            if s not in cache:
                cache[s] = load_rows(tokenizer, K.download(C.BANSPEECH_REPO, [bs_files[s]]), "audio", "transcription")
            rows += sample_rows(cache[s], C.TEST_CLIPS_PER_SPLIT, seed=seed + k, phone=phone)
        return rows

    tests = {
        "test_subakko_clean": sk_test,                                                  # standard Bangla, studio
        "test_subakko_phone": [simulate(r, 5000 + i) for i, r in enumerate(sk_test)],   # the same clips, phone line
        "test_bs_spont_phone": banspeech(C.BANSPEECH_SPONT, phone=True),                # conversation, phone line
        "test_bs_spont_clean": banspeech(C.BANSPEECH_SPONT, phone=False),
        "test_bs_medicine_phone": banspeech(C.BANSPEECH_MEDICINE, phone=True),          # medical words, phone line
        "test_bs_clean": banspeech(C.BANSPEECH_CLEAN, phone=False),                     # news + audiobooks
    }
    for name, rows in tests.items():
        save_rows(tokenizer, rows, out / "tests" / name, name)
    report["tests"] = {k: len(v) for k, v in tests.items()}

    print("\n[4/4] Phone simulation example")
    phone_sim_picture(tests["test_bs_spont_clean"], out / "examples")

    (out / "data_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    settings_file.write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding="utf-8")   # marks "done"
    if C.DELETE_RAW_AFTER_PREP:
        shutil.rmtree(C.raw_dir(), ignore_errors=True)
        print(f"deleted downloads in {C.raw_dir()} (DELETE_RAW_AFTER_PREP)")
    print(f"\nData ready in {out} ({(time.time() - t_start) / 60:.1f} min). Report: {out / 'data_report.json'}")


def parser():
    p = K.base_parser("Step 1: download + prepare data")
    p.add_argument("--inspect", action="store_true", help="only download one training file + show what is in it")
    p.add_argument("--force", action="store_true", help="rebuild even if prepared data exists")
    return p


if __name__ == "__main__":
    a = parser().parse_args()
    K.setup(a, "prepare_data")
    main(a)

In [ ]:
%%writefile baseline.py
"""
STEP 2: score the ORIGINAL model on all test sets, before any training.

  python baseline.py            # skips if the baseline already exists for this data
  python baseline.py --force

Without a baseline you cannot tell whether training helped. Decoding = production settings
(beam 5, no_repeat_ngram_size 3, repetition_penalty 1.2), exactly like moonshine_base_bn/moonshine_bn.py.

Output (shared by all runs on the same prepared data):
  data/processed/baseline_scores.csv
  data/processed/baseline_hyps.json     (transcripts, used for side-by-side comparison later)
"""

import json

import common as K
import config as C


def main(args):
    out = C.processed_dir()
    csv, hyp_file = out / "baseline_scores.csv", out / "baseline_hyps.json"
    if csv.exists() and hyp_file.exists() and not args.force:
        print(f"Baseline already exists: {csv} (use --force to recompute)")
        print(open(csv, encoding="utf-8").read())
        return

    sets = K.load_eval_sets()
    tokenizer = K.load_tokenizer()
    model = K.load_model()
    print(f"device: {K.device()} | precision for decoding: {K.precision()}")
    df, hyps = K.evaluate_sets(model, tokenizer, sets, "baseline")
    df.to_csv(csv, index=False)
    hyp_file.write_text(json.dumps(hyps, ensure_ascii=False, indent=1), encoding="utf-8")
    print("\n" + df.to_string(index=False))
    own = K.own_test_files()
    if own:
        print(f"\nYour own calls ({len(own)} labelled files in {C.OWN_TEST_DIR}), for information "
              "(evaluate_model.py scores them again with both models):")
        K.evaluate_own(model, tokenizer, "baseline", own)
    else:
        print(f"\nNo labelled calls in {C.OWN_TEST_DIR} yet (x.wav + x.txt). See README section 3.0.")

    name = "test_bs_spont_phone"
    if name in sets:
        print(f"\nExamples from {name} (what the phone problem looks like):")
        for (_, ref), hyp in list(zip(sets[name], hyps[name]))[:5]:
            print("  REF:", ref)
            print("  HYP:", hyp, "\n")
    print(f"saved: {csv}")


def parser():
    p = K.base_parser("Step 2: baseline scores of the original model")
    p.add_argument("--force", action="store_true", help="recompute even if it exists")
    return p


if __name__ == "__main__":
    a = parser().parse_args()
    K.setup(a, "baseline")
    main(a)

In [ ]:
%%writefile train.py
"""
STEP 3: fine-tune.

  python train.py                 # starts, or RESUMES automatically from the last checkpoint of this run
  python train.py --fresh         # ignore existing checkpoints and start over

What happens (details in README section 2):
  * every step: BATCH_SIZE clips -> collator (phone simulation, padding, attention mask, labels with -100)
    -> model(audio, labels) -> cross-entropy loss -> backprop -> AdamW update
  * gradients of GRAD_ACCUM batches are added before one update (effective batch = BATCH_SIZE * GRAD_ACCUM)
  * learning rate: linear warm-up over the first 10% of updates, then linear decay to 0
  * every EVAL_STEPS updates: generate text for the dev set (greedy), compute WER/CER, save a checkpoint
  * at the end: reload the checkpoint with the LOWEST dev WER (not necessarily the last) and save it as
    runs/<run>/final_model  (+ fixed generation_config.json, tokenizer, training_info.json)

If the process is stopped (disconnect, crash, Ctrl+C), just run it again: it continues from the last
checkpoint (saved every EVAL_STEPS updates, so at most EVAL_STEPS updates are lost).
"""

import json
import math
import shutil
import time

import numpy as np

import common as K
import config as C


def plot_curves(hist, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    if "loss" in hist:
        tr = hist.dropna(subset=["loss"])
        ax[0].plot(tr["step"], tr["loss"], label="train loss")
    if "eval_loss" in hist:
        ev = hist.dropna(subset=["eval_loss"])
        ax[0].plot(ev["step"], ev["eval_loss"], "o-", label="dev loss")
        ax[1].plot(ev["step"], ev["eval_wer"], "o-", label="dev WER")
        ax[1].plot(ev["step"], ev["eval_cer"], "o-", label="dev CER")
    for a in ax:
        a.set_xlabel("update step")
        a.legend()
        a.grid(alpha=.3)
    ax[0].set_title("loss")
    ax[1].set_title("dev error rate (greedy decoding)")
    plt.tight_layout()
    plt.savefig(path, dpi=120)
    plt.close(fig)


def main(args):
    import pandas as pd
    import torch
    import transformers
    from datasets import load_from_disk
    from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback
    from transformers.trainer_utils import get_last_checkpoint

    data = C.processed_dir()
    if not (data / "prep_settings.json").exists():
        raise SystemExit(f"No prepared data in {data}. Run prepare_data.py first.")
    run = C.run_dir()
    ckpt_dir = run / "checkpoints"
    if args.fresh and ckpt_dir.exists():
        shutil.rmtree(ckpt_dir)

    # numpy format: audio comes out as arrays (fast) instead of Python lists
    train_ds = load_from_disk(str(data / "train")).with_format("numpy")
    dev_ds = load_from_disk(str(data / "dev")).with_format("numpy")
    tokenizer = K.load_tokenizer()
    model = K.load_model()
    prec = K.precision()

    steps_per_epoch = math.ceil(len(train_ds) / (C.BATCH_SIZE * C.GRAD_ACCUM))
    total = C.MAX_STEPS if C.MAX_STEPS > 0 else steps_per_epoch * C.NUM_EPOCHS
    gpu = torch.cuda.get_device_name(0) if K.device() == "cuda" else "none (CPU, very slow)"
    print(f"GPU: {gpu} | precision: {prec} | transformers {transformers.__version__}")
    print(f"train: {len(train_ds)} clips ({np.sum(train_ds['length']) / K.SR / 3600:.1f} h) | dev: {len(dev_ds)} clips")
    print(f"batch {C.BATCH_SIZE} x accum {C.GRAD_ACCUM} = {C.BATCH_SIZE * C.GRAD_ACCUM} clips per update | "
          f"{steps_per_epoch} updates/epoch | {total} updates in total | LR {C.LEARNING_RATE}")

    def compute_metrics(pred):
        p = np.where(pred.predictions < 0, K.PAD_ID, pred.predictions)
        l = np.where(pred.label_ids < 0, K.PAD_ID, pred.label_ids)
        hyps = [K.normalize_text(s) for s in tokenizer.batch_decode(p, skip_special_tokens=True)]
        refs = [K.normalize_text(s) for s in tokenizer.batch_decode(l, skip_special_tokens=True)]
        w, c = K.score(refs, hyps)
        return {"wer": w, "cer": c}

    class SaveHistory(TrainerCallback):
        """Write log_history.csv + curves.png after every evaluation, so you can watch progress."""

        def on_evaluate(self, a, state, control, **kw):
            hist = pd.DataFrame(state.log_history)
            hist.to_csv(run / "log_history.csv", index=False)
            try:
                plot_curves(hist, run / "curves.png")
            except Exception as e:
                print("plot failed:", e)

    targs = Seq2SeqTrainingArguments(
        output_dir=str(ckpt_dir),
        per_device_train_batch_size=C.BATCH_SIZE,
        per_device_eval_batch_size=C.EVAL_BATCH_SIZE,
        gradient_accumulation_steps=C.GRAD_ACCUM,
        learning_rate=C.LEARNING_RATE,
        lr_scheduler_type="linear",
        warmup_steps=C.WARMUP,
        weight_decay=C.WEIGHT_DECAY,
        max_grad_norm=1.0,
        num_train_epochs=C.NUM_EPOCHS,
        max_steps=C.MAX_STEPS,
        fp16=(prec == "fp16"),
        bf16=(prec == "bf16"),
        eval_strategy="steps", eval_steps=C.EVAL_STEPS,
        save_strategy="steps", save_steps=C.EVAL_STEPS, save_total_limit=C.SAVE_TOTAL_LIMIT,
        load_best_model_at_end=True, metric_for_best_model="wer", greater_is_better=False,
        predict_with_generate=True, generation_max_length=160, generation_num_beams=1,
        logging_steps=25, logging_first_step=True, report_to="none",
        remove_unused_columns=False,
        train_sampling_strategy="group_by_length", length_column_name="length",
        dataloader_num_workers=C.NUM_WORKERS,
        disable_tqdm=False,
        seed=C.SEED,
    )
    trainer = Seq2SeqTrainer(model=model, args=targs, train_dataset=train_ds, eval_dataset=dev_ds,
                             data_collator=K.SpeechCollator(C.PHONE_SIM_PROB, seed=C.SEED),
                             compute_metrics=compute_metrics, processing_class=tokenizer,
                             callbacks=[SaveHistory()])

    last = get_last_checkpoint(str(ckpt_dir)) if ckpt_dir.exists() else None
    print("resuming from:", last) if last else print("starting fresh")
    t0 = time.time()
    trainer.train(resume_from_checkpoint=last)
    minutes = (time.time() - t0) / 60
    print(f"training time: {minutes:.1f} min")
    print(f"best checkpoint: {trainer.state.best_model_checkpoint} | best dev WER (greedy): {trainer.state.best_metric}")

    hist = pd.DataFrame(trainer.state.log_history)
    hist.to_csv(run / "log_history.csv", index=False)
    plot_curves(hist, run / "curves.png")

    final = run / "final_model"
    trainer.save_model(str(final))
    tokenizer.save_pretrained(str(final))
    K.fixed_generation_config().save_pretrained(str(final))
    (final / "training_args.bin").unlink(missing_ok=True)
    report = json.loads((data / "data_report.json").read_text(encoding="utf-8"))
    info = {
        "base_model": str(C.BASE_MODEL), "run_name": C.RUN_NAME, "quick_test": C.QUICK,
        "train_clips": report.get("train_clips"), "train_hours": report.get("train_hours"),
        "train_by_source": report.get("train"), "data_settings": C.data_settings(),
        "phone_sim_prob": C.PHONE_SIM_PROB, "learning_rate": C.LEARNING_RATE, "epochs": C.NUM_EPOCHS,
        "max_steps": C.MAX_STEPS, "batch": C.BATCH_SIZE, "grad_accum": C.GRAD_ACCUM, "warmup": C.WARMUP,
        "precision": prec, "gpu": gpu, "training_minutes": round(minutes, 1),
        "best_checkpoint": trainer.state.best_model_checkpoint, "best_dev_wer_greedy": trainer.state.best_metric,
        "versions": {"transformers": transformers.__version__, "torch": torch.__version__},
    }
    (final / "training_info.json").write_text(json.dumps(info, ensure_ascii=False, indent=2, default=str),
                                              encoding="utf-8")
    print(f"final model saved: {final}")


def parser():
    p = K.base_parser("Step 3: fine-tune (resumes automatically)")
    p.add_argument("--fresh", action="store_true", help="delete this run's checkpoints and start over")
    return p


if __name__ == "__main__":
    a = parser().parse_args()
    K.setup(a, "train")
    main(a)

In [ ]:
%%writefile evaluate_model.py
"""
STEP 4: score the fine-tuned model and compare with the baseline.

  python evaluate_model.py              # uses runs/<run>/final_model
  python evaluate_model.py --zip        # also make final_model.zip (to copy the model to another machine)

Same test sets and the same production decoding (beam 5) as baseline.py, so the numbers are comparable.

Output in runs/<run>/:
  final_scores.csv     baseline + fine-tuned rows
  results.md           before/after table, examples, your own recordings side by side

How to read the table (README section 3.4): the most important row is own_test_set (YOUR labelled calls in
own_test_set/). Without it, look at the *_phone rows. The clean rows (test_subakko_clean, test_bs_clean) must
not get worse by more than ~1-2 WER points.
"""

import json
import shutil
from pathlib import Path

import pandas as pd

import common as K
import config as C


def main(args):
    run = C.run_dir()
    final = run / "final_model"
    if not (final / "model.safetensors").exists():
        raise SystemExit(f"No fine-tuned model in {final}. Run train.py first.")
    base_csv = C.processed_dir() / "baseline_scores.csv"
    if not base_csv.exists():
        raise SystemExit("No baseline scores. Run baseline.py first.")

    sets = K.load_eval_sets()
    tokenizer = K.load_tokenizer()
    model = K.load_model(final)
    print(f"device: {K.device()} | model: {final}")
    ft_df, ft_hyps = K.evaluate_sets(model, tokenizer, sets, "finetuned")
    base_df = pd.read_csv(base_csv)
    base_hyps = json.loads((C.processed_dir() / "baseline_hyps.json").read_text(encoding="utf-8"))

    # your own labelled calls: scored with BOTH models here (you may add files after the baseline was made)
    own = K.own_test_files()
    my_dir = Path(C.MY_AUDIO_DIR)
    files = sorted(p for p in my_dir.iterdir() if p.suffix.lower() in K.AUDIO_EXTS) if my_dir.is_dir() else []
    base_model = K.load_model() if (own or files) else None
    if own:
        print(f"\nown_test_set: {len(own)} labelled files in {C.OWN_TEST_DIR}")
        b_row, own_base = K.evaluate_own(base_model, tokenizer, "baseline", own)
        f_row, own_ft = K.evaluate_own(model, tokenizer, "finetuned", own)
        base_df = pd.concat([base_df, pd.DataFrame([b_row])])
        ft_df = pd.concat([ft_df, pd.DataFrame([f_row])])
    else:
        print(f"(no labelled files in {C.OWN_TEST_DIR}: the most important test is missing, see README 3.0)")

    both =pd.concat([base_df, ft_df])
    both.to_csv(run / "final_scores.csv", index=False)
    cmp = both.pivot(index="test_set", columns="model", values=["WER", "CER"])
    table = pd.DataFrame({
        "WER before": cmp[("WER", "baseline")], "WER after": cmp[("WER", "finetuned")],
        "WER change": cmp[("WER", "finetuned")] - cmp[("WER", "baseline")],
        "CER before": cmp[("CER", "baseline")], "CER after": cmp[("CER", "finetuned")],
        "CER change": cmp[("CER", "finetuned")] - cmp[("CER", "baseline")],
    }).round(4)
    print("\n" + table.to_string())

    md = [f"# Results: `{C.RUN_NAME}`", "", "Production decoding (beam 5). Negative change = better.", "",
          table.reset_index().to_markdown(index=False) if _has_tabulate() else "```\n" + table.to_string() + "\n```", ""]
    info_file = final / "training_info.json"
    if info_file.exists():
        info = json.loads(info_file.read_text(encoding="utf-8"))
        md += [f"Training: {info.get('train_hours')} h, LR {info.get('learning_rate')}, epochs {info.get('epochs')}, "
               f"{info.get('training_minutes')} min on {info.get('gpu')}, best dev WER (greedy) "
               f"{info.get('best_dev_wer_greedy')}.", ""]
        info["scores"] = both.to_dict("records")
        info_file.write_text(json.dumps(info, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

    if own:
        md += ["## Your own calls (own_test_set)", ""]
        for (p, ref), b, f in zip(own, own_base, own_ft):
            (wb, _), (wf, _) = K.score([ref], [b]), K.score([ref], [f])
            md += [f"- **{p.name}** (WER {wb:.2f} → {wf:.2f})", f"  - ref: {ref}", f"  - before: {b}",
                   f"  - after: {f}"]
        md.append("")

    name = "test_bs_spont_phone"
    if name in sets and name in base_hyps:
        md += [f"## Examples: {name}", ""]
        for (_, ref), b, f in list(zip(sets[name], base_hyps[name], ft_hyps[name]))[:10]:
            md += [f"- **ref:** {ref}", f"  - before: {b}", f"  - after: {f}"]
        md.append("")

    # ---- recordings in sample-audio, old vs new model (scored only if a .txt is next to the audio)
    if files:
        print(f"\nOwn recordings in {C.MY_AUDIO_DIR}:")
        md += ["## Own recordings (sample-audio)", ""]
        for p in files:
            b = K.transcribe_file(base_model, tokenizer, p)
            f = K.transcribe_file(model, tokenizer, p)
            md += [f"### {p.name}", "", f"- **before:** {b}", f"- **after:** {f}"]
            print(f"\n== {p.name}\n  before: {b}\n  after : {f}")
            ref = p.with_suffix(".txt")
            if ref.exists():
                r = K.normalize_text(ref.read_text(encoding="utf-8"))
                (wb, cb), (wf, cf) = K.score([r], [b]), K.score([r], [f])
                md.append(f"- WER {wb:.3f} → {wf:.3f}, CER {cb:.3f} → {cf:.3f}")
                print(f"  WER {wb:.3f} -> {wf:.3f} | CER {cb:.3f} -> {cf:.3f}")
            md.append("")
    else:
        print(f"(no audio in {C.MY_AUDIO_DIR}; skipping sample-audio recordings)")

    (run / "results.md").write_text("\n".join(md), encoding="utf-8")
    print(f"\nreport: {run / 'results.md'}")
    if args.zip:
        z = shutil.make_archive(str(run / "final_model"), "zip", str(final))
        print(f"zip: {z}")


def _has_tabulate():
    try:
        import tabulate  # noqa: F401  (pandas needs it for to_markdown)
        return True
    except ImportError:
        return False


def parser():
    p = K.base_parser("Step 4: evaluate the fine-tuned model vs the baseline")
    p.add_argument("--zip", action="store_true", help="also create runs/<run>/final_model.zip")
    return p


if __name__ == "__main__":
    a = parser().parse_args()
    K.setup(a, "evaluate")
    main(a)

In [ ]:
%%writefile run_all.py
"""
Run all steps in order: prepare_data -> baseline -> train -> evaluate.

  python run_all.py --quick       # tiny end-to-end check first (~10-15 min on a GPU)
  python run_all.py               # the real run

Every step skips work that is already done (prepared data, baseline) and training resumes from the last
checkpoint, so after a crash or disconnect you just run the same command again.

On a remote Linux machine, run it so it survives closing the terminal:
  nohup python run_all.py > run_all.out 2>&1 &      (watch with: tail -f run_all.out)
or inside `tmux` / `screen`.
"""

import baseline
import common as K
import evaluate_model
import prepare_data
import train


def main():
    p = K.base_parser("Run all fine-tuning steps")
    p.add_argument("--force-data", action="store_true", help="rebuild the prepared data")
    p.add_argument("--fresh", action="store_true", help="restart training from scratch (delete checkpoints)")
    p.add_argument("--zip", action="store_true", help="zip the final model at the end")
    a = p.parse_args()
    K.setup(a, "run_all")

    a.force, a.inspect = a.force_data, False
    print("\n##### STEP 1/4: prepare data #####")
    prepare_data.main(a)
    a.force = False
    print("\n##### STEP 2/4: baseline #####")
    baseline.main(a)
    print("\n##### STEP 3/4: train #####")
    train.main(a)
    print("\n##### STEP 4/4: evaluate #####")
    evaluate_model.main(a)
    print("\nAll done.")


if __name__ == "__main__":
    main()

In [ ]:
# per-platform settings -> config_local.py (overrides config.py; same mechanism as on a local GPU PC)
local = f"""
DATA_DIR = Path({str(DATA)!r})
RAW_DIR = Path({str(Path('/tmp/moonshine_ft_raw') if ON_KAGGLE else Path('/content/ft_raw') if ON_COLAB else DATA / 'raw')!r})
DELETE_RAW_AFTER_PREP = {bool(ON_KAGGLE or ON_COLAB)}
RUNS_DIR = Path({str(RUNS)!r})
OWN_TEST_DIR = Path({str(OWN)!r}) if {OWN is not None} else HERE / "own_test_set"
MY_AUDIO_DIR = HERE / "sample-audio"
RUN_NAME = {RUN_NAME!r}
TRAIN_SOURCES[0].update(files=subakko_train_files(max(1, min(47, round({TRAIN_HOURS} / 3)))), hours={TRAIN_HOURS})
PHONE_SIM_PROB = {PHONE_SIM_PROB}
NUM_EPOCHS = {NUM_EPOCHS}
LEARNING_RATE = {LEARNING_RATE}
BATCH_SIZE, GRAD_ACCUM = {BATCH_SIZE}, {GRAD_ACCUM}
"""
Path("config_local.py").write_text(local, encoding="utf-8")
print(local)
Q = "--quick" if QUICK else ""
print("mode:", "QUICK test" if QUICK else "REAL run")

## 5. Step 1: prepare the data
Downloads SUBAK.KO train files, the SUBAK.KO validation/test files and BanSpeech, cleans the text, drops clips
that do not fit our rules (digits, English letters, too long...), and builds:
`train/`, `dev/`, `tests/*`. It prints example texts (raw → cleaned label), sample rates and how many clips were
dropped and why. Real run: ~10 GB download, ~20-40 min. Skipped automatically if already built.

In [ ]:
!python prepare_data.py $Q

In [ ]:
# listen: a clean clip and the same clip through the simulated phone line
from IPython.display import Audio, Image, display
import config as C
ex = Path(C.DATA_DIR) / ("processed_quick" if QUICK else "processed") / "examples"
for w in sorted(ex.glob("*.wav"))[:4]:
    print(w.name); display(Audio(str(w)))
if (ex / "phone_sim_spectrogram.png").exists():
    display(Image(str(ex / "phone_sim_spectrogram.png")))

## 6. Step 2: baseline (the original model, before training)
Without this you cannot tell whether training helped. Uses the production decoding (beam 5). Lower WER/CER =
better. Also scores your own labelled calls if they were found.

In [ ]:
!python baseline.py $Q

## 7. Step 3: train
Every 250 updates: dev WER/CER + a checkpoint. If the session stops, run all cells again: it **resumes** from
the last checkpoint. Watch `eval_wer` going down. Train loss going down while dev WER goes up = overfitting
(the best checkpoint is kept anyway).

In [ ]:
!python train.py $Q

In [ ]:
import config as C
run = Path(C.RUNS_DIR) / ("quick_test" if QUICK else RUN_NAME)
if (run / "curves.png").exists():
    display(Image(str(run / "curves.png")))

## 8. Step 4: evaluate and compare
Same test sets + same decoding as the baseline. Read the table like this:
- **own_test_set**: the real answer (your calls). Lower after = success.
- `*_phone` rows should go down (that is what stage 1 trains for).
- `test_subakko_clean` / `test_bs_clean` must not get worse by more than ~1-2 WER points.

In [ ]:
!python evaluate_model.py $Q --zip

In [ ]:
from IPython.display import Markdown
res = run / "results.md"
if res.exists():
    display(Markdown(res.read_text(encoding="utf-8")))

## 9. Download the model
`final_model.zip` (~230 MB) is in the run folder:
- **Colab:** it is on Google Drive (`MyDrive/moonshine_ft/runs/<run>/`); or run the cell below.
- **Kaggle:** after "Save Version" it is in the notebook's *Output* tab (`runs/<run>/final_model.zip`).

On your PC unzip it to `Fine-tune-moonshine/models/<run>/` and compare with
`python Fine-tune-moonshine/test_finetuned.py --model Fine-tune-moonshine/models/<run>` (README section 3.6).

In [ ]:
z = run / "final_model.zip"
print(z, z.exists())
if ON_COLAB and z.exists():
    from google.colab import files
    files.download(str(z))